In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import glob
import copy
import os

In [3]:
class Subnetwork(nn.Module):
    def __init__(self, input_dim, output_dim = 1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        return self.net(x)

    def train(model, dataloader, epochs = 2000, patience = 80, min_delta = 1e-6):
        criterion = nn.L1Loss()
        optimizer = optim.Adam(model.parameters(), lr=5e-4)
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma = 0.95)

        best_loss = float('inf')
        patience_counter = 0
        best_weights = copy.deepcopy(model.state_dict())

        model.train()
        print(f"Trainig has been started...\n Epochs: {epochs} \n Patience: {patience}")

        for epoch in range(epochs):
            epoch_loss = 0

            for u_seq, y_true_seq in dataloader:
                x_init = torch.zeros(u_seq.size(0), model.num_states)
                optimizer.zero_grad()
                y_pred_seq = model(u_seq, x_init)
                loss = criterion(y_pred_seq.view_as(y_true_seq), y_true_seq)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
            
            scheduler.step()
            avg_loss = epoch_loss / len(dataloader)

            if (epoch + 1) % 10 == 0:
                print(f"Epoch: {epoch+1}/{epochs} | MAE: {avg_loss:.6f}")

            if best_loss - avg_loss > min_delta:
                best_loss = avg_loss               
                patience_counter = 0               
                best_model_weights = copy.deepcopy(model.state_dict()) 
            else:
                patience_counter += 1            
            
            if patience_counter >= patience:
                print(f"\n[!] EARLY STOPPING: Stopped in {epoch+1} epoch.")
                print(f"No better than {min_delta} since {patience} epochs.")
                break 
            
        model.load_state_dict(best_model_weights)
        print(f"Training done. Best weights with MAE loss: {best_loss:.6f} loaded.\n")
        return model
        
    def create_sequences(inputs, targets, seq_len = 600):
        xs, ys = [], []
        for i in range(0, len(inputs) - seq_len+1, seq_len):
            xs.append(inputs[i : i + seq_len])
            ys.append(targets[i : i + seq_len])
        return np.array(xs), np.array(ys)


    def validate_and_plot(model, scaler_u, scaler_y, input_cols, target_cols, file_path, title_prefix, alpha=0.05, threshold=None, burn_in=0):
        df_test = pd.read_csv(file_path)
        X_test_tensor = torch.tensor(scaler_u.transform(df_test[input_cols].values), dtype=torch.float32).unsqueeze(0)
        Y_test_tensor = torch.tensor(scaler_y.transform(df_test[target_cols].values), dtype=torch.float32).unsqueeze(0)
    
    # Przewidywanie modelu
        model.eval()
        with torch.no_grad():
            y_pred = model(X_test_tensor, torch.zeros(1, model.num_states))
        
            y_pred_real = scaler_y.inverse_transform(y_pred[0].cpu().numpy().reshape(-1, 1))
            y_true_real = scaler_y.inverse_transform(Y_test_tensor[0].cpu().numpy().reshape(-1, 1))
            czas = np.arange(len(y_true_real)) * 0.05
    
    # ---------------------------------------------------------
    # OBLICZANIE I FILTROWANIE REZIDUUM (BŁĘDU BEZWZGLĘDNEGO)
    # ---------------------------------------------------------
        e_raw = np.abs(y_true_real - y_pred_real) # Surowy błąd (z Twojego kodu)
        e_filtered = np.zeros_like(e_raw)         # Pusta tablica na przefiltrowany błąd
        e_filtered[0] = e_raw[0]                  # Stan początkowy
    
    # Pętla filtru EMA (Exponential Moving Average - Filtr dolnoprzepustowy I rzędu)
        for i in range(1, len(e_raw)):
            e_filtered[i] = alpha * e_raw[i] + (1 - alpha) * e_filtered[i-1]
        
    # --- NAPRAWA BŁĘDU STANU POCZĄTKOWEGO (Burn-in period) ---
    # Sztuczne wyzerowanie błędu dla pierwszych N próbek (np. 50 próbek = 2.5 sekundy), 
    # żeby model zdążył się numerycznie "napompować" do wartości fizycznych.
        if burn_in > 0:
            e_raw[:burn_in] = 0.0
            e_filtered[:burn_in] = 0.0
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    
    # Górny wykres: Fizyka
        ax1.plot(czas, y_true_real, label='Prawda (Czujnik)', color='black', alpha=0.7)
        ax1.plot(czas, y_pred_real, label='Symulacja (AI)', color='red', linestyle='--', alpha=0.9)
        ax1.set_title(f'{title_prefix} | Plik: {file_path.split("/")[-1]}', fontsize=12)
        ax1.set_ylabel('Wartość fizyczna (np. Pa)')
        ax1.legend(loc='upper right')
        ax1.grid(True, linestyle=':', alpha=0.7)
    
    # Dolny wykres: Diagnostyka
        ax2.plot(czas, e_raw, label='Surowe Reziduum (Szum)', color='gray', alpha=0.4)
        ax2.plot(czas, e_filtered, label=f'Przefiltrowane Reziduum (alpha={alpha})', color='darkred', linewidth=2)
    
    # Jeśli podałeś próg alarmowy, narysuj go
        if threshold is not None:
            ax2.axhline(threshold, color='orange', linestyle='--', linewidth=2, label=f'Próg Detekcji ({threshold})')
        
        ax2.set_title('Sygnał Diagnostyczny (Reziduum)', fontsize=12)
        ax2.set_xlabel('Czas [s]')
        ax2.set_ylabel('Błąd predykcji')
        ax2.legend(loc='upper right')
        ax2.grid(True, linestyle=':', alpha=0.7)
    
        plt.tight_layout()
        plt.show()
    
    # Do liczenia statystyk bierzemy dane dopiero PO fazie rozgrzewki, 
    # żeby ten gigantyczny spadek początkowy nie niszczył wyników MAE!
        print(f"Średni błąd bezwzględny (MAE po rozgrzewce): {np.mean(e_raw[burn_in:]):.4f}")
        print(f"Maksymalny błąd przefiltrowany (po rozgrzewce): {np.max(e_filtered[burn_in:]):.4f}")

In [ ]:
class GreyBoxSystem_WAF()